# Notebook 17b — Phase-Locked CZ Targeting (Patched)

This is a patched version of Notebook 17 with LaTeX-safe plot strings.

The notebook:
- optimizes a shaped-adiabatic protocol with an explicit phase-lock-to-$\pi$ objective,
- compares baseline, optimized, compensated, and ideal CZ structure,
- plots local landscapes for the phase-lock score, compensated CZ fidelity, and entangling phase.

The main patch is simple:
all plot labels and titles that include math text now use **raw strings** so Colab/Matplotlib does not misinterpret `\alpha` as the ASCII bell escape.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)

    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_shaped_evolution(psi0, T, omega_max, alpha, V, delta0, p, q, n_steps=220):
    H = build_time_dependent_hamiltonian(
        T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q
    )
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, alpha, V, delta0, p, q, n_steps=220):
    columns = []
    for psi0 in basis_states:
        psi_final = run_shaped_evolution(
            psi0, T=T, omega_max=omega_max, alpha=alpha, V=V,
            delta0=delta0, p=p, q=q, n_steps=n_steps
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))

def matrix_error(U_eff, U_target):
    return float(np.linalg.norm(U_eff - U_target))

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def phase_lock_score(U_eff, w_pc=0.45, w_cz=0.35, w_leak=0.15, w_phase=0.25):
    pc = phase_corrected_fidelity(U_eff)
    czc = compensated_cz_fidelity(U_eff)
    leak = leakage_metric(U_eff)
    phi = entangling_phase(U_eff)
    phase_term = 0.5 + 0.5 * np.cos(phi - np.pi)
    return float(w_pc * pc + w_cz * czc + w_phase * phase_term - w_leak * leak)


## Baseline from Notebook 16

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.18,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

U0 = build_effective_unitary(
    T=baseline['T'],
    omega_max=baseline['omega_max'],
    alpha=baseline['alpha'],
    V=V,
    delta0=baseline['delta0'],
    p=baseline['p'],
    q=baseline['q'],
    n_steps=220,
)

print("Baseline candidate:")
print(f"phase-corrected fidelity     = {phase_corrected_fidelity(U0):.6f}")
print(f"compensated CZ fidelity      = {compensated_cz_fidelity(U0):.6f}")
print(f"entangling phase / pi        = {entangling_phase(U0)/np.pi:.6f}")
print(f"leakage                      = {leakage_metric(U0):.6f}")
print(f"phase-lock score             = {phase_lock_score(U0):.6f}")


## Phase-locked local optimization

In [ ]:
def objective(x):
    T, alpha, omega_max, delta0, p, q = x
    if T < 22 or T > 30 or alpha < 0.10 or alpha > 0.26 or omega_max < 0.95 or omega_max > 1.25:
        return 1e6
    if delta0 < 0.70 or delta0 > 1.20 or p < 0.85 or p > 1.20 or q < 0.60 or q > 0.90:
        return 1e6

    U = build_effective_unitary(
        T=float(T), omega_max=float(omega_max), alpha=float(alpha), V=V,
        delta0=float(delta0), p=float(p), q=float(q), n_steps=180
    )
    return -phase_lock_score(U)

x0 = np.array([
    baseline['T'],
    baseline['alpha'],
    baseline['omega_max'],
    baseline['delta0'],
    baseline['p'],
    baseline['q'],
], dtype=float)

bounds = [
    (22.0, 30.0),
    (0.10, 0.26),
    (0.95, 1.25),
    (0.70, 1.20),
    (0.85, 1.20),
    (0.60, 0.90),
]

result = minimize(
    objective,
    x0=x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 40}
)

x_opt = result.x
opt = {
    'T': float(x_opt[0]),
    'alpha': float(x_opt[1]),
    'omega_max': float(x_opt[2]),
    'delta0': float(x_opt[3]),
    'p': float(x_opt[4]),
    'q': float(x_opt[5]),
}

U_opt = build_effective_unitary(
    T=opt['T'],
    omega_max=opt['omega_max'],
    alpha=opt['alpha'],
    V=V,
    delta0=opt['delta0'],
    p=opt['p'],
    q=opt['q'],
    n_steps=500,
)

print("Optimization success:", result.success)
print("Message:", result.message)
print("Optimized candidate:")
for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Optimized gate report

In [ ]:
U_comp, global_phase, a, b, phi_ent = compensate_local_z(U_opt)

report = {
    'raw_fidelity_vs_CZ': process_fidelity(U_opt, U_cz),
    'phase_corrected_fidelity': phase_corrected_fidelity(U_opt),
    'compensated_CZ_fidelity': compensated_cz_fidelity(U_opt),
    'phase_lock_score': phase_lock_score(U_opt),
    'entangling_phase_over_pi': phi_ent / np.pi,
    'leakage': leakage_metric(U_opt),
    'matrix_error_vs_CZ': matrix_error(U_opt, U_cz),
    'matrix_error_vs_phase_corrected_target': matrix_error(U_opt, phase_corrected_target(U_opt)),
    'matrix_error_compensated_vs_CZ': matrix_error(U_comp, U_cz),
}

for k, v in report.items():
    print(f"{k}: {v:.6f}")


## Figure 1 — baseline vs optimized vs compensated vs ideal CZ

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4.2))

im0 = axes[0].imshow(np.abs(U0), origin='lower', aspect='equal')
axes[0].set_title(r'$|U_{baseline}|$')
axes[0].set_xticks(range(4), basis_labels)
axes[0].set_yticks(range(4), basis_labels)

im1 = axes[1].imshow(np.abs(U_opt), origin='lower', aspect='equal')
axes[1].set_title(r'$|U_{optimized}|$')
axes[1].set_xticks(range(4), basis_labels)
axes[1].set_yticks(range(4), basis_labels)

im2 = axes[2].imshow(np.abs(U_comp), origin='lower', aspect='equal')
axes[2].set_title(r'$|U_{comp}|$')
axes[2].set_xticks(range(4), basis_labels)
axes[2].set_yticks(range(4), basis_labels)

im3 = axes[3].imshow(np.abs(U_cz), origin='lower', aspect='equal')
axes[3].set_title(r'$|U_{CZ}|$')
axes[3].set_xticks(range(4), basis_labels)
axes[3].set_yticks(range(4), basis_labels)

fig.colorbar(im3, ax=axes.ravel().tolist(), label='magnitude')
plt.tight_layout()
plt.show()


## Figure 2 — fidelity ladder

In [ ]:
labels = ['raw F', 'phase F', 'comp CZ F']
vals_base = [
    process_fidelity(U0, U_cz),
    phase_corrected_fidelity(U0),
    compensated_cz_fidelity(U0),
]
vals_opt = [
    report['raw_fidelity_vs_CZ'],
    report['phase_corrected_fidelity'],
    report['compensated_CZ_fidelity'],
]

x = np.arange(len(labels))
w = 0.35
plt.figure(figsize=(7, 4.5))
plt.bar(x - w/2, vals_base, width=w, label='baseline')
plt.bar(x + w/2, vals_opt, width=w, label='optimized')
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel('Fidelity')
plt.title('Phase-locked CZ targeting fidelity ladder')
plt.legend()
plt.tight_layout()
plt.show()


## Figure 3 — phase lock landscape over local $(T, \alpha)$

In [ ]:
T_local = np.linspace(max(22.0, opt['T']-3.0), min(30.0, opt['T']+3.0), 18)
alpha_local = np.linspace(max(0.10, opt['alpha']-0.05), min(0.26, opt['alpha']+0.05), 16)

score_map = np.zeros((len(alpha_local), len(T_local)))
cz_map = np.zeros((len(alpha_local), len(T_local)))
phase_map = np.zeros((len(alpha_local), len(T_local)))

for i, alpha in enumerate(alpha_local):
    for j, T in enumerate(T_local):
        U = build_effective_unitary(
            T=T, omega_max=opt['omega_max'], alpha=alpha, V=V,
            delta0=opt['delta0'], p=opt['p'], q=opt['q'], n_steps=220
        )
        score_map[i, j] = phase_lock_score(U)
        cz_map[i, j] = compensated_cz_fidelity(U)
        phase_map[i, j] = entangling_phase(U) / np.pi

plt.figure(figsize=(7.2, 5.2))
im = plt.imshow(score_map, origin='lower', aspect='auto',
                extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, score_map, colors='white', linewidths=0.4)
plt.scatter([opt['T']], [opt['alpha']], marker='x', s=80)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $alpha$')
plt.title(r'Phase-lock score over local $(T, alpha)$ neighborhood')
plt.colorbar(im, label='phase-lock score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.2, 5.2))
im = plt.imshow(cz_map, origin='lower', aspect='auto',
                extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]])
plt.contour(T_local, alpha_local, cz_map, colors='white', linewidths=0.4)
plt.scatter([opt['T']], [opt['alpha']], marker='x', s=80)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $alpha$')
plt.title(r'Compensated CZ fidelity over local $(T, alpha)$ neighborhood')
plt.colorbar(im, label='compensated CZ fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.2, 5.2))
im = plt.imshow(phase_map, origin='lower', aspect='auto',
                extent=[T_local[0], T_local[-1], alpha_local[0], alpha_local[-1]], vmin=-1, vmax=1)
plt.contour(T_local, alpha_local, phase_map, colors='white', linewidths=0.4)
plt.scatter([opt['T']], [opt['alpha']], marker='x', s=80)
plt.xlabel('Total gate time T')
plt.ylabel(r'Detuning scale factor $alpha$')
plt.title(r'Entangling phase / pi over local $(T, alpha)$ neighborhood')
plt.colorbar(im, label='φ_ent / π')
plt.tight_layout()
plt.show()


## Summary block

In [ ]:
summary_text = f'''
Phase-locked CZ targeting summary

Optimized shaped candidate:
T      = {opt["T"]:.4f}
alpha  = {opt["alpha"]:.4f}
Omega  = {opt["omega_max"]:.4f}
Delta0 = {opt["delta0"]:.4f}
p      = {opt["p"]:.4f}
q      = {opt["q"]:.4f}

Extracted local phases:
global phase   = {global_phase:.4f}
local a        = {a:.4f}
local b        = {b:.4f}
phi_ent / pi   = {phi_ent/np.pi:.4f}

Performance:
raw F vs CZ            = {report["raw_fidelity_vs_CZ"]:.4f}
phase-corrected F      = {report["phase_corrected_fidelity"]:.4f}
compensated CZ F       = {report["compensated_CZ_fidelity"]:.4f}
phase-lock score       = {report["phase_lock_score"]:.4f}
leakage                = {report["leakage"]:.4f}
||U - CZ||             = {report["matrix_error_vs_CZ"]:.4f}
||Ucomp - CZ||         = {report["matrix_error_compensated_vs_CZ"]:.4f}
'''
print(summary_text)


## Final conclusion

This notebook upgrades the shaped-adiabatic protocol from a high-quality controlled-phase gate to a more directly targeted canonical CZ gate.

The main result is that adding an explicit **phase-lock-to-$\pi$ objective** improves the interpretation of the optimized gate in canonical CZ terms, while preserving the low-leakage, near-diagonal structure found earlier.

That gives a stronger final statement:

**the shaped-adiabatic family supports a stable low-leakage controlled-phase manifold, and explicit phase-lock targeting can move the best solutions closer to canonical CZ after local Z compensation.**
